# 当日开盘状态固定增量训练
两组同为执行收益标签；对照含缺失标记，候选另加两资产隔夜跳空及前15分钟收益。训练6个模型，回测6个完整账户，固定0/2/5bp，保留标签单独模型及长期参考。

In [1]:
from pathlib import Path
import os,json,qlib
os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
root=Path('/Users/rziiiii/dev/workspace');card=root/'research/opening_state_training_20261015'
qlib.init(provider_uri=str(root/'data/qlib/full_path_tiered_allocation_20261013'),region='us')
from src.workflow.opening_state_training import run
summary=run(root=root,card=card,plan=json.loads((card/'plan.json').read_text()))
print(summary)


# 当日开盘状态增量训练：不晋升

完成6次真实拟合、6套新增Qlib原生账户；复用18套参考账户。两组均使用1056条成熟主训练标签（2018-10-12至2022-12-21，标签成熟截止2023-01-01）和20741条辅助数据，3个seed、300轮固定末轮等权集成。主标签均为09:47至下一交易日09:47执行收益，原H5分类标签不变。

配对对照输入原39字段加开盘数据可用标记；候选再加SOXL/SOXX隔夜跳空及0930至0945收益共4字段。标记作为双方共同基础输入标准化；新增数值系数零初始化，初始输出配对一致。数值仅用训练期已知观察拟合缩放器，未知值只在标准化模型通道置零，不改原始数据。已知开盘信息通过前交易日收盘和当日0945已结束bar构造，决策时点0946，当前0947成交价和当日收盘不进入输入。时间资格仍是历史bar加60秒的研究假设，不是实时首收认证。

回测为2023-01-03信号至2026-10-09退出，共944区间，本金10万美元，IBKR Pro Tiered冻结费用投影，各滑点账户独立持仓、费用与风险状态。

| 账户 | 0 bp净收益 | 2 bp净收益 | 5 bp净收益 | 2 bp监控最大回撤 |
|---|---:|---:|---:|---:|
| 同缺失标记对照 | +159.40% | +133.08% | +93.49% | 29.90% |
| 开盘数值候选 | +102.24% | +139.55% | +118.44% | 31.21% |
| 无开盘输入的执行标签模型 | +125.67% | +121.05% | +120.30% | 32.40% |
| 长历史冻结小模型 | +342.52% | +335.36% | +348.06% | 27.64% |
| 同短样本旧标签模型 | +376.04% | +370.06% | +428.74% | 26.35% |
| SOXX | +387.75% | +387.73% | +387.70% | 43.39% |

候选0bp低于配对对照，0/5bp低于无开盘输入的执行标签模型，三个成本均远低于长期强参考和SOXX；监控回撤31.00%/31.21%/31.56%均超过30%。独立成本下风险状态路径不同，不能从收益非单调性推出更高滑点有益。超额、风险及稳定门失败；已反复使用的历史不是新的独立样本。

四个数值字段均在1054条已知训练样本上激活，三个seed训练后相对logit系数均非零；参数数目由301增至313，初始logits最大差为0。训练期已知样本数及训练后相对logit系数见feature_usage.csv；非零系数仅证明被使用，不代表预测能力。保留缺失日期，包括已有SOXX首分钟异常，不填造数据。共享标记导致初始化与无标记旧模型不同，因此配对数值增量只由本轮两组判断，旧模型只作外部强参考。价格调整/成交价/监控时点依旧是既定历史研究代理，未认证实际券商成交、真实派息或全分钟最大回撤。

决策not_promoted_opening_state_increment_failed。关闭这组固定开盘信息路线，不删字段、不换开盘长度、不扫seed/epoch或控制阈值。下一项信息需求是定位完整账户风险状态为何把小幅预测差异放大为成本敏感的持仓分歧；先从已存逐日账户和订单作归因，再决定是否存在可验证的机制修复，不能凭本次结果修改风控。生产模型未替换，稳定超额目标未完成。

执行notebook、训练初始化审计、模型和原生账户已保存Qlib Recorder，NAS逐对象冷恢复哈希通过。新增信息时点/缺失合同测试1项通过。
